# GRN inference

Infers transcription factor (TF) → target gene edges for two stimulus contexts, K562 + TNF-α and THP1 + LPS. Each context uses three steps:

1. **Rank TFs** (`cp.tl.grn.rank_regulators`): signed diffusion over the OmniPath signaling network from the context's upstream signal. TFs (human TFs with OmniPath transcriptional targets) absorb the mass that reaches them and are ranked by the net activating mass.
2. **Score target genes** (`cp.tl.grn.score_targets`): convert the context's tradeSeq association p-values into local FDRs (fdrtool / beta-uniform mixture) over protein-coding genes.
3. **Infer edges** (`cp.tl.grn.infer`): weight each TF → target edge by the TF's priority and the target's 1 - local FDR, then rank the edges.

Results for each context are written to `outputs/<context>/`.


In [ ]:
from pathlib import Path

import chronopack as cp

HUMAN_TFS = "/new-stg/home/alex/ref/gene_sets/hs_tfs.txt"
LIGAND_RECEPTORS = "/new-stg/home/alex/ref/cellchatdb/hs/interaction_input_CellChatDB.csv"
PROTEIN_CODING = "/new-stg/home/alex/tmp/grn_inference_v2/reproduction/inputs/external/ref/genome/hg38/annotations/protein_coding_genes.csv"
ANALYSES = Path("/new-stg/home/alex/tmp/grn_inference_v2/reproduction/inputs/external/projects/chrono_seq/analyses")


## K562 + TNF-α

TFs are ranked downstream of the TNF ligand (seeded at its receptors); target genes come from the K562 tradeSeq association test.

In [ ]:
tfs_k562 = cp.tl.grn.rank_regulators(
    source="TNF",
    is_ligand=True,
    human_tf_path=HUMAN_TFS,
    ligand_receptor_path=LIGAND_RECEPTORS,
)
tfs_k562.head(10)


In [ ]:
targets_k562 = cp.tl.grn.score_targets(
    ANALYSES / "k562-tnfa.hg38.rep/intmd/sc-K562-1e4_to_tradeSeq/asso_df.1trajectory.lfdr.csv",
    valid_targets=PROTEIN_CODING,
)
targets_k562.summary


In [ ]:
grn_k562 = cp.tl.grn.infer(targets_k562, tfs_k562, valid_targets=PROTEIN_CODING, name="ChronoPack_K562")
grn_k562.edges.head(10)


In [ ]:
targets_k562.write("outputs/K562", prefix="target_scores")
tfs_k562.to_csv("outputs/K562/tf_ranking.csv", index=False)
grn_k562.write("outputs/K562", prefix="grn")


## THP1 + LPS

TFs are ranked downstream of the TLR4 receptor; target genes come from the THP1 tradeSeq association test.

In [ ]:
tfs_thp1 = cp.tl.grn.rank_regulators(
    source="TLR4",
    human_tf_path=HUMAN_TFS,
    ligand_receptor_path=LIGAND_RECEPTORS,
)
tfs_thp1.head(10)


In [ ]:
targets_thp1 = cp.tl.grn.score_targets(
    ANALYSES / "thp1-k562-lps.hg38.rep/intmd/sc-THP1_to_tradeSeq/asso_df.1trajectory.lfdr.csv",
    valid_targets=PROTEIN_CODING,
)
targets_thp1.summary


In [ ]:
grn_thp1 = cp.tl.grn.infer(targets_thp1, tfs_thp1, valid_targets=PROTEIN_CODING, name="ChronoPack_THP1")
grn_thp1.edges.head(10)


In [ ]:
targets_thp1.write("outputs/THP1", prefix="target_scores")
tfs_thp1.to_csv("outputs/THP1/tf_ranking.csv", index=False)
grn_thp1.write("outputs/THP1", prefix="grn")
